# ATLAS hydro scenario change analysis

This notebook applies CMIP6 climate change signals to the present-day river discharge climatology derived from GLOFAS.

The workflow is designed to work for countries with station integration, such as Ecuador, and for countries without station integration, such as Bolivia. The only difference is controlled by the parameter `USE_STATIONS`.

The notebook loads preprocessed CMIP6 historical and future scenario datasets, loads present-day GLOFAS basin statistics at HydroBASINS level 12, and then applies monthly CMIP6 percentage changes to the present-day river discharge baseline.

Since CMIP6 river discharge projections have a spatial resolution of approximately 100 km, while GLOFAS data are available at about 5.5 km resolution, CMIP6 data are first interpolated onto the GLOFAS grid. Because river discharge is a non-linear variable and interpolation can create artefacts outside the river network, the interpolated CMIP6 fields are clipped using HydroRIVERS and a GLOFAS discharge threshold.

Monthly percentage change is computed by comparing the future SSP experiment with the CMIP6 historical experiment. The percentage change is then applied to the present-day GLOFAS basin climatology. If station-integrated data are available, the delta is applied to the station-integrated column. If stations are not available, the delta is applied to the original GLOFAS basin statistics.

Final outputs are saved at HydroBASINS level 12 and aggregated to HydroBASINS level 5, including both future river discharge and delta river discharge.

## Step 1. Import libraries

This step loads the Python libraries used to read NetCDF files, manage geospatial layers, compute zonal statistics and save the final outputs.

In [1]:
from pathlib import Path
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray  # Required to use the .rio accessor on xarray objects.
import rasterio as rio
import rasterstats as rstats

warnings.filterwarnings("ignore")

ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


## Step 2. Define user parameters

Edit only this cell to change country, model, scenario, period or folders.

Set `USE_STATIONS = True` when the present-day level 12 basin statistics include the station-integrated column `mean_river_discharge_integrated`.

Set `USE_STATIONS = False` when station data are not available and the notebook should use the original GLOFAS basin statistics column `mean_river_discharge`.

In [2]:
# ---------------------------------------------------------------------
# Main configuration
# ---------------------------------------------------------------------

# Choose one of the predefined countries below, or edit the fields manually.
# Ecuador usually uses station-integrated basin statistics.
# Bolivia usually uses basin statistics without station integration.

COUNTRY_FOLDER = "bolivia"      # Examples: "ecuador", "bolivia"
COUNTRY_NAME_IN_SHAPEFILE = "Bolivia"

USE_STATIONS = False             # True for Ecuador, False for Bolivia if station data are not available.

# CMIP6 variable name. For river discharge, use "rivo".
VARIABLE = "rivo"
VARIABLE_LONG_NAME = "river_discharge"

MODEL = "CNRM-ESM2-1"
HISTORICAL_EXPERIMENT = "historical"
SCENARIO_EXPERIMENT = "ssp370"

# Scenario period used to compute the future monthly climatology.
SCENARIO_START = "2020"
SCENARIO_END = "2050"

# Periods used in the filenames produced by the preprocessing notebooks.
GLOFAS_START = "1991-01"
GLOFAS_END = "2020-12"
HISTORICAL_START = "1985-01"
HISTORICAL_END = "2014-12"
SCENARIO_FILE_START = "2015-01"
SCENARIO_FILE_END = "2100-12"

# Grid cells with present GLOFAS discharge lower than or equal to this value are ignored.
RIVER_DISCHARGE_THRESHOLD = 1

# Months to process.
MONTHS = range(1, 13)

# ---------------------------------------------------------------------
# Input folders and files
# ---------------------------------------------------------------------

GLOFAS_FILE = (
    Path("../data/processed")
    / VARIABLE_LONG_NAME
    / COUNTRY_FOLDER
    / f"river_discharge_{GLOFAS_START}_{GLOFAS_END}_processed.nc"
)

CMIP6_PROCESSED_DIR = Path("../data/processed") / VARIABLE_LONG_NAME / COUNTRY_FOLDER

HISTORICAL_FILE = (
    CMIP6_PROCESSED_DIR
    / MODEL
    / HISTORICAL_EXPERIMENT
    / f"{VARIABLE}_{HISTORICAL_START}_{HISTORICAL_END}_processed.nc"
)

SCENARIO_FILE = (
    CMIP6_PROCESSED_DIR
    / MODEL
    / SCENARIO_EXPERIMENT
    / f"{VARIABLE}_{SCENARIO_FILE_START}_{SCENARIO_FILE_END}_processed.nc"
)

# Present-day basin statistics at HydroBASINS level 12.
# With USE_STATIONS = True, the notebook expects files named:
# river_discharge_l12_{country}_m{month}_integrated.csv
#
# With USE_STATIONS = False, the notebook expects files named:
# river_discharge_l12_{country}_m{month}.csv

BASIN_STATISTICS_DIR = Path("../data/atlas_data") / COUNTRY_FOLDER

# ---------------------------------------------------------------------
# Output folder
# ---------------------------------------------------------------------

OUTPUT_DIR = (
    Path("../data/atlas_data")
    / COUNTRY_FOLDER
    / MODEL
    / SCENARIO_EXPERIMENT
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Static geospatial inputs
# ---------------------------------------------------------------------

SHAPEFILE_PATH = Path("../world_map/ne_50m_admin_0_countries.shp")
HYDRO_RIVERS_PATH = Path("../hydrorivers/HydroRIVERS_v10_sa_shp/HydroRIVERS_v10_sa.shp")
HYDROBASINS_DIR = Path("../hydrobasins")

print("Country folder:", COUNTRY_FOLDER)
print("Country name in shapefile:", COUNTRY_NAME_IN_SHAPEFILE)
print("Use station-integrated data:", USE_STATIONS)
print("GLOFAS file:", GLOFAS_FILE)
print("Historical CMIP6 file:", HISTORICAL_FILE)
print("Scenario CMIP6 file:", SCENARIO_FILE)
print("Input basin statistics folder:", BASIN_STATISTICS_DIR)
print("Output folder:", OUTPUT_DIR)

Country folder: bolivia
Country name in shapefile: Bolivia
Use station-integrated data: False
GLOFAS file: ../data/processed/river_discharge/bolivia/river_discharge_1991-01_2020-12_processed.nc
Historical CMIP6 file: ../data/processed/river_discharge/bolivia/CNRM-ESM2-1/historical/rivo_1985-01_2014-12_processed.nc
Scenario CMIP6 file: ../data/processed/river_discharge/bolivia/CNRM-ESM2-1/ssp370/rivo_2015-01_2100-12_processed.nc
Input basin statistics folder: ../data/atlas_data/bolivia
Output folder: ../data/atlas_data/bolivia/CNRM-ESM2-1/ssp370


## Step 3. Define helper functions

These functions keep the workflow shorter and easier to read. They also make the same methodology work for both Ecuador and Bolivia by selecting the correct present-day discharge column automatically.

In [3]:
def read_country_geometry(shapefile_path, country_name):
    """Read the selected country geometry from the Natural Earth shapefile."""
    countries = gpd.read_file(shapefile_path)[["ADMIN", "geometry"]]
    countries = countries.set_index("ADMIN")
    if countries.crs is None:
        countries = countries.set_crs("EPSG:4326")
    countries = countries.to_crs("EPSG:4326")
    if country_name not in countries.index:
        raise ValueError(f"Country '{country_name}' was not found in {shapefile_path}")
    return countries.loc[[country_name]]


def retrieve_hydrobasins(level, country_geometry):
    """Load HydroBASINS for one level and clip it to the selected country."""
    level_string = str(level).zfill(2)
    basins_file = HYDROBASINS_DIR / f"hybas_lake_sa_lev{level_string}_v1c.shp"

    basins = gpd.read_file(basins_file).to_crs("EPSG:4326")

    lakes = basins[basins["LAKE"] == 1].copy()
    basins = basins[basins["LAKE"] == 0].copy()

    country_basins = gpd.clip(basins, country_geometry.geometry)
    country_lakes = gpd.clip(lakes, country_geometry.geometry)

    return country_basins, country_lakes


def retrieve_country_rivers(rivers_path, country_geometry):
    """Load HydroRIVERS and keep only river reaches inside the selected country."""
    rivers = gpd.read_file(rivers_path).to_crs("EPSG:4326")
    return gpd.clip(rivers, country_geometry.geometry)


def prepare_dataset_for_spatial_operations(ds):
    """Ensure that an xarray dataset has spatial dimensions and EPSG:4326 CRS."""
    ds = ds.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
    if ds.rio.crs is None:
        ds = ds.rio.write_crs("EPSG:4326", inplace=False)
    return ds


def get_monthly_climatology(ds, variable_name, start=None, end=None):
    """Select an optional time period and compute monthly climatological means."""
    if start is not None or end is not None:
        ds = ds.sel(time=slice(start, end))
    return ds[variable_name].groupby("time.month").mean(skipna=True)


def interpolate_to_glofas_grid(cmip_monthly_da, glofas_monthly_da, method="nearest"):
    """Interpolate CMIP6 monthly data to the GLOFAS grid."""
    return cmip_monthly_da.interp(
        latitude=glofas_monthly_da.latitude.values,
        longitude=glofas_monthly_da.longitude.values,
        method=method,
    )


def clip_dataarray_to_rivers(da, rivers_gdf, variable_name):
    """Clip a DataArray using the selected river geometries."""
    ds = da.to_dataset(name=variable_name)
    ds = prepare_dataset_for_spatial_operations(ds)

    clipped = ds.rio.clip(
        rivers_gdf.geometry.values,
        rivers_gdf.crs,
        all_touched=True,
    )

    return clipped[variable_name]


def mask_cmip_using_glofas(cmip_da, glofas_da, threshold):
    """Keep CMIP6 values only where GLOFAS discharge is above the selected threshold."""
    glofas_aligned, cmip_aligned = xr.align(glofas_da, cmip_da, join="right")
    return cmip_aligned.where(glofas_aligned > threshold)


def compute_percent_change(historical_da, scenario_da):
    """Compute monthly percentage change between scenario and historical CMIP6."""
    historical_aligned, scenario_aligned = xr.align(historical_da, scenario_da)

    delta = scenario_aligned - historical_aligned

    percent_change = xr.where(
        historical_aligned != 0,
        (delta / historical_aligned) * 100,
        np.nan,
    )

    percent_change.name = "rivo_percent_change"

    return percent_change.to_dataset()


def get_l12_input_file(month):
    """Return the expected level 12 basin statistics file for the selected month."""
    if USE_STATIONS:
        filename = f"river_discharge_l12_{COUNTRY_FOLDER}_m{month}_integrated.csv"
    else:
        filename = f"river_discharge_l12_{COUNTRY_FOLDER}_m{month}.csv"

    return BASIN_STATISTICS_DIR / filename


def read_l12_basin_statistics(month):
    """Read one monthly level 12 basin statistics CSV and return a GeoDataFrame."""
    input_file = get_l12_input_file(month)

    if not input_file.exists():
        raise FileNotFoundError(
            f"Missing input file: {input_file}\n"
            "Check COUNTRY_FOLDER, USE_STATIONS and BASIN_STATISTICS_DIR."
        )

    df = pd.read_csv(input_file)

    if "Unnamed: 0" in df.columns:
        df = df.drop(columns="Unnamed: 0")

    if "geometry" not in df.columns:
        raise ValueError(
            f"The file {input_file} does not contain a 'geometry' column. "
            "The basin statistics notebook must save geometry as WKT."
        )

    df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"])

    return gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")


def select_present_discharge_column(gdf):
    """Find the present-day river discharge column to use as baseline."""
    if USE_STATIONS and "mean_river_discharge_integrated" in gdf.columns:
        return "mean_river_discharge_integrated"

    if "mean_river_discharge" in gdf.columns:
        return "mean_river_discharge"

    if "mean_river_discharge_integrated" in gdf.columns:
        return "mean_river_discharge_integrated"

    raise ValueError(
        "No valid present-day river discharge column was found. "
        "Expected 'mean_river_discharge_integrated' or 'mean_river_discharge'."
    )


def get_affine_from_reference_file(reference_file):
    """Read the raster affine transform from the reference GLOFAS NetCDF file."""
    with rio.open(reference_file) as src:
        return src.transform


def aggregate_percent_change_on_basins(percent_change_ds, country_basins, month, affine):
    """Aggregate monthly percentage change over HydroBASINS polygons."""
    gdf = country_basins.copy()

    xdf_month = percent_change_ds.sel(month=month).drop_vars("month")
    variable_name = list(xdf_month.data_vars)[0]

    mean_basins = rstats.zonal_stats(
        gdf.geometry,
        xdf_month[variable_name].values,
        affine=affine,
        stats="mean",
    )

    gdf["mean_percent_change"] = [stat["mean"] for stat in mean_basins]
    gdf["mean_percent_change"] = gdf["mean_percent_change"].fillna(0)

    return gdf


def apply_percent_delta_to_reanalysis(percent_change_l12, present_l12):
    """Apply CMIP6 percentage change to the present-day GLOFAS basin discharge."""
    present_column = select_present_discharge_column(present_l12)

    delta = percent_change_l12[["HYBAS_ID", "mean_percent_change"]].copy()

    gdf_future = present_l12.merge(delta, on="HYBAS_ID", how="left")
    gdf_future["mean_percent_change"] = gdf_future["mean_percent_change"].fillna(0)

    gdf_future["present_river_discharge"] = gdf_future[present_column]

    gdf_future["mean_river_discharge_future"] = (
        gdf_future["present_river_discharge"]
        * (1 + gdf_future["mean_percent_change"] / 100)
    )

    gdf_future["delta_river_discharge"] = (
        gdf_future["mean_river_discharge_future"]
        - gdf_future["present_river_discharge"]
    )

    columns_to_keep = [
        "HYBAS_ID",
        "PFAF_ID",
        "SUB_AREA",
        "present_river_discharge",
        "mean_percent_change",
        "mean_river_discharge_future",
        "delta_river_discharge",
        "geometry",
    ]

    available_columns = [column for column in columns_to_keep if column in gdf_future.columns]

    return gdf_future[available_columns]


def aggregate_l12_to_l5(gdf_l12, country_basins_l5):
    """Aggregate level 12 scenario values to level 5 using SUB_AREA as weight."""
    gdf_l12 = gdf_l12.to_crs(country_basins_l5.crs)

    gdf_points = gdf_l12.copy()
    gdf_points["geometry"] = gdf_points.geometry.representative_point()

    joined = gpd.sjoin(
        gdf_points,
        country_basins_l5[["HYBAS_ID", "geometry"]],
        how="left",
        predicate="within",
    )

    joined["weighted_present_q"] = joined["present_river_discharge"] * joined["SUB_AREA"]
    joined["weighted_future_q"] = joined["mean_river_discharge_future"] * joined["SUB_AREA"]
    joined["weighted_delta_q"] = joined["delta_river_discharge"] * joined["SUB_AREA"]
    joined["weighted_percent_change"] = joined["mean_percent_change"] * joined["SUB_AREA"]

    aggregated = (
        joined
        .groupby("HYBAS_ID_right")
        .agg(
            weighted_present_q=("weighted_present_q", "sum"),
            weighted_future_q=("weighted_future_q", "sum"),
            weighted_delta_q=("weighted_delta_q", "sum"),
            weighted_percent_change=("weighted_percent_change", "sum"),
            area=("SUB_AREA", "sum"),
        )
        .reset_index()
    )

    aggregated["present_river_discharge"] = aggregated["weighted_present_q"] / aggregated["area"]
    aggregated["mean_river_discharge_future"] = aggregated["weighted_future_q"] / aggregated["area"]
    aggregated["delta_river_discharge"] = aggregated["weighted_delta_q"] / aggregated["area"]
    aggregated["mean_percent_change"] = aggregated["weighted_percent_change"] / aggregated["area"]

    aggregated = aggregated.rename(columns={"HYBAS_ID_right": "HYBAS_ID"})

    return country_basins_l5.merge(
        aggregated[
            [
                "HYBAS_ID",
                "present_river_discharge",
                "mean_percent_change",
                "mean_river_discharge_future",
                "delta_river_discharge",
            ]
        ],
        on="HYBAS_ID",
        how="left",
    )


def save_geodataframe_to_csv(gdf, output_file):
    """Save a GeoDataFrame to CSV with geometry in WKT format."""
    output_file.parent.mkdir(parents=True, exist_ok=True)

    df = gdf.copy()
    df["geometry"] = df.geometry.to_wkt()
    df.to_csv(output_file, index=False)


def save_geodataframe_to_geojson(gdf, output_file):
    """Save a GeoDataFrame to GeoJSON."""
    output_file.parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(output_file, driver="GeoJSON")

## Step 4. Load country, rivers and basins

This step loads the country boundary, HydroRIVERS and HydroBASINS. HydroRIVERS is used to reduce interpolation noise outside the river network.

In [ ]:
country_geometry = read_country_geometry(SHAPEFILE_PATH, COUNTRY_NAME_IN_SHAPEFILE)

country_rivers = retrieve_country_rivers(HYDRO_RIVERS_PATH, country_geometry)
rivers_only = country_rivers[country_rivers["ORD_FLOW"] < 7].copy()

country_basins_l5, country_lakes_l5 = retrieve_hydrobasins(5, country_geometry)
country_basins_l12, country_lakes_l12 = retrieve_hydrobasins(12, country_geometry)

country_basins_l12["L5basin"] = [
    int(str(pfaf_id)[:5]) for pfaf_id in country_basins_l12["PFAF_ID"]
]

print("Country boundary:", country_geometry.total_bounds)
print("Number of river segments used as mask:", len(rivers_only))
print("Number of level 5 basins:", len(country_basins_l5))
print("Number of level 12 basins:", len(country_basins_l12))

## Step 5. Load GLOFAS and CMIP6 datasets

This step opens the processed GLOFAS and CMIP6 NetCDF files and computes monthly climatologies. The future scenario is clipped to the selected scenario period.

In [ ]:
glofas_ds = xr.open_dataset(GLOFAS_FILE)
glofas_ds = prepare_dataset_for_spatial_operations(glofas_ds)
glofas_monthly = get_monthly_climatology(glofas_ds, variable_name="dis24")

historical_ds = xr.open_dataset(HISTORICAL_FILE)
historical_monthly = get_monthly_climatology(
    historical_ds,
    variable_name=VARIABLE,
)

scenario_ds = xr.open_dataset(SCENARIO_FILE)
scenario_monthly = get_monthly_climatology(
    scenario_ds,
    variable_name=VARIABLE,
    start=SCENARIO_START,
    end=SCENARIO_END,
)

print("GLOFAS monthly climatology:")
print(glofas_monthly)
print("\nHistorical CMIP6 monthly climatology:")
print(historical_monthly)
print("\nScenario CMIP6 monthly climatology:")
print(scenario_monthly)

## Step 6. Interpolate CMIP6 to the GLOFAS grid

CMIP6 river discharge is much coarser than GLOFAS. The notebook interpolates CMIP6 monthly climatologies to the GLOFAS grid before comparing historical and future experiments.

In [ ]:
historical_monthly_interp = interpolate_to_glofas_grid(
    cmip_monthly_da=historical_monthly,
    glofas_monthly_da=glofas_monthly,
    method="nearest",
)

scenario_monthly_interp = interpolate_to_glofas_grid(
    cmip_monthly_da=scenario_monthly,
    glofas_monthly_da=glofas_monthly,
    method="nearest",
)

print("Interpolated historical CMIP6:")
print(historical_monthly_interp)
print("\nInterpolated scenario CMIP6:")
print(scenario_monthly_interp)

## Step 7. Mask CMIP6 using the present river network

The interpolated CMIP6 fields are masked using HydroRIVERS and the present-day GLOFAS discharge threshold. This reduces spurious values produced by interpolation outside the river network.

In [ ]:
glofas_monthly_masked = clip_dataarray_to_rivers(
    da=glofas_monthly,
    rivers_gdf=rivers_only,
    variable_name="dis24",
)

historical_monthly_masked = mask_cmip_using_glofas(
    cmip_da=historical_monthly_interp,
    glofas_da=glofas_monthly_masked,
    threshold=RIVER_DISCHARGE_THRESHOLD,
)

scenario_monthly_masked = mask_cmip_using_glofas(
    cmip_da=scenario_monthly_interp,
    glofas_da=glofas_monthly_masked,
    threshold=RIVER_DISCHARGE_THRESHOLD,
)

print("Masked historical CMIP6:")
print(historical_monthly_masked)
print("\nMasked scenario CMIP6:")
print(scenario_monthly_masked)

## Step 8. Compute monthly CMIP6 percentage change

For each month, the notebook computes:

`percentage change = (scenario - historical) / historical * 100`

This percentage change is later applied to the present-day GLOFAS basin statistics.

In [ ]:
percent_change_monthly = compute_percent_change(
    historical_da=historical_monthly_masked,
    scenario_da=scenario_monthly_masked,
)

print(percent_change_monthly)

## Step 9. Apply CMIP6 changes to present-day basin statistics

For each month, this step aggregates the CMIP6 percentage change over HydroBASINS level 12, reads the present-day GLOFAS basin statistics, applies the percentage change, saves level 12 outputs and aggregates the result to level 5.

The same code works with or without station integration. With `USE_STATIONS = True`, the baseline column is `mean_river_discharge_integrated`. With `USE_STATIONS = False`, the baseline column is `mean_river_discharge`.

In [ ]:
affine = get_affine_from_reference_file(GLOFAS_FILE)

for month in MONTHS:
    print("\n" + "=" * 80)
    print(f"Processing month {month}")

    percent_change_l12 = aggregate_percent_change_on_basins(
        percent_change_ds=percent_change_monthly,
        country_basins=country_basins_l12,
        month=month,
        affine=affine,
    )

    present_l12 = read_l12_basin_statistics(month)

    scenario_l12 = apply_percent_delta_to_reanalysis(
        percent_change_l12=percent_change_l12,
        present_l12=present_l12,
    )

    scenario_l5 = aggregate_l12_to_l5(
        gdf_l12=scenario_l12,
        country_basins_l5=country_basins_l5,
    )

    l12_csv = OUTPUT_DIR / (
        f"river_discharge_l12_{COUNTRY_FOLDER}_m{month}_"
        f"{SCENARIO_EXPERIMENT}_{SCENARIO_START}_{SCENARIO_END}.csv"
    )

    l5_csv = OUTPUT_DIR / (
        f"river_discharge_l5_{COUNTRY_FOLDER}_m{month}_"
        f"{SCENARIO_EXPERIMENT}_{SCENARIO_START}_{SCENARIO_END}.csv"
    )

    l5_geojson = OUTPUT_DIR / (
        f"river_discharge_l5_{COUNTRY_FOLDER}_m{month}_"
        f"{SCENARIO_EXPERIMENT}_{SCENARIO_START}_{SCENARIO_END}.geojson"
    )

    save_geodataframe_to_csv(scenario_l12, l12_csv)
    save_geodataframe_to_csv(scenario_l5, l5_csv)
    save_geodataframe_to_geojson(scenario_l5, l5_geojson)

    print("Saved level 12 CSV:", l12_csv)
    print("Saved level 5 CSV:", l5_csv)
    print("Saved level 5 GeoJSON:", l5_geojson)

## Step 10. Check the outputs

This final step lists the files created in the output folder.

In [ ]:
output_files = sorted(OUTPUT_DIR.glob("*"))

print(f"Number of output files: {len(output_files)}")
for output_file in output_files:
    print(output_file)